In [1]:
import os

import os
import torch
import numpy as np
from math import *
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
def DFT_matrix(N):
    i, j = np.meshgrid(np.arange(N), np.arange(N))
    omega = np.exp(- 2 * pi * 1J / N)
    W = np.power(omega, i * j) / sqrt(N)
    return np.mat(W)


In [3]:
def freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt):
    #Nc代表载波数 N代表路径数 sigma_2_alph每条路径能量 sigma_angle角度均匀分布的幅度
    d=0.5
    #print(check)
    Tau = Nc  #最大路径时延
    tau = np.random.rand(1,N)*Tau
    if isinstance(Nt,int):
        n_r = np.mat(range(Nr)).reshape(Nr,1)
        n_t = np.mat(range(Nt)).reshape(Nt,1)
        fai1 = np.random.rand(1,N)*2*pi
        A_t = np.mat(np.zeros([Nt,N],dtype = complex))
        A_r = np.mat(np.ones([1,N],dtype = complex))
        alpha = np.squeeze(np.zeros([1,N],dtype = complex),0)
        H_f = np.array(np.zeros([1,Nc,Nt],dtype = complex))
        for i in range(N):
            A_t[:,i] = np.exp(-2j*pi*0.5*n_t*np.sin(fai1[0,i]))
            aa = (np.random.randn(1,1)+1j*np.random.randn(1,1))*np.sqrt(sigma_2_alpha/2/N)
            alpha[i] = aa[0,0]
    else:
        N_r = Nr
        N_t = Nt[0]*Nt[1]
        fait = np.random.rand(1,N)*2*pi
        fair = np.random.rand(1,N)*2*pi
        theatt = np.random.rand(1,N)*2*pi
        theatr = np.random.rand(1,N)*2*pi
        A_t = np.mat(np.zeros([N_t,N],dtype = complex))
        A_r = np.mat(np.zeros([N_r,N],dtype = complex))
        alpha = np.squeeze(np.zeros([1,N],dtype = complex),0)
        H_f = np.array(np.zeros([N_r,N_t,Nc],dtype = complex))
        n_t1 = np.mat(range(Nt[0])).reshape(Nt[0],1)
        n_t2 = np.mat(range(Nt[1])).reshape(Nt[1],1)
        A_r = np.mat(np.ones([1,N],dtype = complex))
        H_f = np.array(np.zeros([1,Nc,N_t],dtype = complex))
        for i in range(N):
            at1 = np.exp(-2j*pi*0.5*n_t1*np.cos(fait[0,i])*np.sin(theatt[0,i]))
            at2 = np.exp(-2j*pi*0.5*n_t2*np.sin(fait[0,i]))
            A_t[:,i] = np.kron(at1,at2)

            aa = (np.random.randn(1,1)+1j*np.random.randn(1,1))*np.sqrt(sigma_2_alpha/2/N)
            alpha[i] = aa[0,0]


    for k in range(Nc):
        P = np.squeeze(alpha*np.exp(-1j*2*pi*tau*k/Nc),0);
        P = np.diag(P)
        H_f[:,k,:] = np.dot(np.dot(A_r,P),A_t.H)

    return H_f,A_t,alpha

In [9]:
# 在 channel_OFDM.ipynb 中替换此函数

# def freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt):
#     # 保持原有的4个输入参数
#     d=0.5
#     Nr = 1 # 内部设定用户天线为1，符合MISO场景
#
#     Tau = Nc
#     tau = np.random.rand(1,N)*Tau
#
#     N_t = Nt[0]*Nt[1]
#
#     # --- 核心修正 1: 角度分布与论文对齐 ---
#     # 论文指定 U(-pi/2, pi/2)，等价于 (np.random.rand() - 0.5) * np.pi
#     fait = (np.random.rand(1, N) - 0.5) * np.pi   # Tx Azimuth (phi in paper)
#     theatt = (np.random.rand(1, N) - 0.5) * np.pi # Tx Elevation (theta in paper)
#
#     A_t = np.mat(np.zeros([N_t,N],dtype = complex))
#     A_r = np.mat(np.ones([Nr,N],dtype = complex)) # A_r for Nr=1 is just a row of ones
#     alpha = np.squeeze(np.zeros([1,N],dtype = complex),0)
#     H_f = np.array(np.zeros([Nr,Nc,N_t],dtype = complex))
#
#     for i in range(N):
#         # --- 核心修正 2: 修正数组维度错误 ---
#         # 使用与论文(5)一致的公式，并确保输出为列向量
#         n_y_indices, n_z_indices = np.meshgrid(range(Nt[0]), range(Nt[1]))
#         term = n_y_indices * np.sin(theatt[0,i]) * np.cos(fait[0,i]) + n_z_indices * np.sin(fait[0,i])
#         at_vec = np.exp(1j * np.pi * term)
#         # .flatten() 产生 (64,) 数组, .reshape(-1, 1) 将其变为 (64, 1) 列向量
#         A_t[:,i] = at_vec.flatten().reshape(-1, 1)
#
#         aa = (np.random.randn(1,1)+1j*np.random.randn(1,1))*np.sqrt(sigma_2_alpha/2/N)
#         alpha[i] = aa[0,0]
#
#     for k in range(Nc):
#         P = np.diag(np.squeeze(alpha*np.exp(-1j*2*pi*tau*k/Nc),0));
#         # H_f for Nr=1 has shape (1, Nc, Nt)
#         H_f[:,k,:] = np.dot(np.dot(A_r,P),A_t.H)
#
#     # 函数的返回值应为 (Nc, Nr, Nt)
#     # H_f is (1, Nc, Nt), we need to squeeze and reshape
#     # 返回 H_f (Nc, Nt) 以匹配原始代码的行为
#     return np.squeeze(H_f, axis=0), A_t, alpha

In [5]:
Nc = 32
sigma_2_alpha = 1
Nt = [8,8]
N_t = Nt[0]*Nt[1]
Nr = 1

B = 30
D = 1000; #角度采样点数
L = 8

SNR_dB = 10
K = 4
snr =  10**(SNR_dB/10)/K

N_BATCH_train = 400
N_BATCH_test  = 80
BATCH_SIZE = 256
N_H_train = N_BATCH_train*BATCH_SIZE
N_H_test = N_BATCH_test*BATCH_SIZE

os.makedirs('data', exist_ok=True)

In [ ]:
for N in range(1,9):
    H_torch = torch.zeros([BATCH_SIZE*N_BATCH_train,K,Nc,N_t*2])
    for i in range(N_BATCH_train):
        H = np.zeros([BATCH_SIZE,K,Nc,N_t],dtype=complex) #第0个维度是样本 第1个维度是用户，第2个维度是子载波，第3个维度是天线
        for j in range(BATCH_SIZE):
            for k in range(K):
                H_f,A_t,alpha = freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt)
                H[j,k,:,:] = H_f
    
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,0:N_t] = torch.from_numpy(np.real(H))
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,N_t:2*N_t] = torch.from_numpy(np.imag(H))
        print(i)
    torch.save(H_torch,'data/H_train_UPA'+str(N)+'Lp_1.pt')
    
    H_torch = torch.zeros([BATCH_SIZE*N_BATCH_train,K,Nc,N_t*2])
    for i in range(N_BATCH_train):
        H = np.zeros([BATCH_SIZE,K,Nc,N_t],dtype=complex) #第0个维度是样本 第1个维度是用户，第2个维度是子载波，第3个维度是天线
        for j in range(BATCH_SIZE):
            for k in range(K):
                H_f,A_t,alpha = freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt)
                H[j,k,:,:] = H_f
    
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,0:N_t] = torch.from_numpy(np.real(H))
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,N_t:2*N_t] = torch.from_numpy(np.imag(H))
        print(i)
    torch.save(H_torch,'data/H_train_UPA'+str(N)+'Lp_2.pt')

In [ ]:
for N in range(1,9):
    H_torch = torch.zeros([BATCH_SIZE*N_BATCH_test,K,Nc,N_t*2])
    for i in range(N_BATCH_test):
        H = np.zeros([BATCH_SIZE,K,Nc,N_t],dtype=complex) #第0个维度是样本 第1个维度是用户，第2个维度是子载波，第3个维度是天线
        for j in range(BATCH_SIZE):
            for k in range(K):
                H_f,A_t,alpha = freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt)
                H[j,k,:,:] = H_f
    
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,0:N_t] = torch.from_numpy(np.real(H))
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,N_t:2*N_t] = torch.from_numpy(np.imag(H))
        print(i)
    torch.save(H_torch,'data/H_test_UPA'+str(N)+'Lp.pt')

In [ ]:

H_torch = torch.zeros([BATCH_SIZE*N_BATCH_train,K,Nc,N_t*2])
for i in range(N_BATCH_train):
    H = np.zeros([BATCH_SIZE,K,Nc,N_t],dtype=complex) #第0个维度是样本 第1个维度是用户，第2个维度是子载波，第3个维度是天线
    for j in range(BATCH_SIZE):
        N = int(np.random.rand(1)[0] * 8) + 1
        for k in range(K):
            H_f,A_t,alpha = freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt)
            H[j,k,:,:] = H_f

    H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,0:N_t] = torch.from_numpy(np.real(H))
    H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,N_t:2*N_t] = torch.from_numpy(np.imag(H))
    print(i)
torch.save(H_torch,'data/H_train_UPA'+str(-1)+'Lp_1.pt')

H_torch = torch.zeros([BATCH_SIZE*N_BATCH_train,K,Nc,N_t*2])
for i in range(N_BATCH_train):
    H = np.zeros([BATCH_SIZE,K,Nc,N_t],dtype=complex) #第0个维度是样本 第1个维度是用户，第2个维度是子载波，第3个维度是天线
    for j in range(BATCH_SIZE):
        N = int(np.random.rand(1)[0] * 8) + 1
        for k in range(K):
            H_f,A_t,alpha = freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt)
            H[j,k,:,:] = H_f
    
    H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,0:N_t] = torch.from_numpy(np.real(H))
    H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,N_t:2*N_t] = torch.from_numpy(np.imag(H))
    print(i)
torch.save(H_torch,'data/H_train_UPA'+str(-1)+'Lp_2.pt')

In [ ]:
H_torch = torch.zeros([BATCH_SIZE*N_BATCH_test,K,Nc,N_t*2])
for i in range(N_BATCH_test):
    H = np.zeros([BATCH_SIZE,K,Nc,N_t],dtype=complex) #第0个维度是样本 第1个维度是用户，第2个维度是子载波，第3个维度是天线
    for j in range(BATCH_SIZE):
        N = int(np.random.rand(1)[0] * 8) + 1
        for k in range(K):
            H_f,A_t,alpha = freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt)
            H[j,k,:,:] = H_f
    
    H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,0:N_t] = torch.from_numpy(np.real(H))
    H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,N_t:2*N_t] = torch.from_numpy(np.imag(H))
    print(i)
torch.save(H_torch,'data/H_test_UPA'+str(-1)+'Lp.pt')

In [10]:
import scipy.io as io

H = np.zeros([BATCH_SIZE,K,Nc,N_t],dtype=complex) #第0个维度是样本 第1个维度是用户，第2个维度是子载波，第3个维度是天线
for j in range(BATCH_SIZE):
    N = int(np.random.rand(1)[0] * 8) + 1
    for k in range(K):
        H_f,A_t,alpha = freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt)
        H[j,k,:,:] = H_f
H_torch = torch.zeros([BATCH_SIZE,K,Nc,N_t*2])
H_torch[:,:,:,0:N_t] = torch.from_numpy(np.real(H))
H_torch[:,:,:,N_t:2*N_t] = torch.from_numpy(np.imag(H))
#torch.save(H_torch,'data/H_test_UPA.pt')
print(Nt)


dataNew = os.path.join('data', 'H_UPA.mat')
io.savemat(dataNew, {'H_UPA': H})

[8, 8]
